# Compositional Distributed Alignment Search for Nested NLI Causal Graphs

This notebook has two parts. **Sections 1-8** are a self-contained DAS replication on a
two-variable *lexical* NLI causal model (lexical relation + monotonicity) using a fine-tuned
GPT-2 and pyvene rotated-subspace interventions. **Sections 9-13** are a faithful replication
of the official **pyvene MQNLI tutorial** (`stanfordnlp/pyvene` advanced tutorials), extended
with a compositional two-variable intervention and a random-init calibration control.

Methodological guardrails:

- lexical DAS chooses a patching site that is actually usable by the counterfactual dataset;
- random-source is treated as a generalization check, not a negative control;
- wrong-variable, shuffled-label, and random-init are the negative/calibration controls;
- monotonicity uses both upward and downward factual templates before DAS;
- **MQNLI follows the tutorial exactly**: a `CausalModel`, a GPT-2 trained with `create_gpt2_lm`
  to predict the relation word, counterfactual data from `generate_counterfactual_dataset`, and
  a `RotatedSpaceIntervention` at a single fixed decision position (`MAX_LENGTH-2`, layer 10).
  DAS numbers are trusted only after the model clears a factual-accuracy gate;
- composition intervenes on two **independent-branch** variables (`NegP`, `NP_S`) at one site
  on orthogonal subspaces, and is calibrated against the joint majority baseline.


## 0. Bootstrap

Run this cell once at the top of every Colab session. It clones the project repo, installs pinned dependencies,
downloads the MQNLI signature JSONs if needed, and creates output folders. Locally, it searches upward for the repo
root and refuses to continue if `nli_das/` is not available.

After this cell finishes:

- `import nli_das` works
- the working directory is the project repo
- `outputs/{figures,tables}/` exist
- all MQNLI signature JSONs are present before the MQNLI section runs


In [ ]:
# Bootstrap
import os, sys, subprocess
from pathlib import Path

REPO_URL  = "https://github.com/aquantumreality/cs221m-final.git"
REPO_NAME = "cs221m-final"

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)])
    os.chdir(repo_root)
else:
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "nli_das").is_dir() and (cand / "requirements.txt").exists():
            os.chdir(cand)
            break
    else:
        raise FileNotFoundError("Could not find cs221m-final repo root containing nli_das/ and requirements.txt.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

try:
    import pyvene  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/stanfordnlp/pyvene.git"])

Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)

TUTORIAL_DATA = Path("tutorial_data")
TUTORIAL_DATA.mkdir(exist_ok=True)
MQNLI_FILES = [
    "mqnli_q_projectivity.json",
    "mqnli_neg_signature.json",
    "mqnli_empty_signature.json",
    "mqnli_cont_signature.json",
    "mqnli_neg_cont_signature.json",
]
BASE = "https://raw.githubusercontent.com/stanfordnlp/pyvene/main/tutorials/advanced_tutorials/tutorial_data/"
missing = [fn for fn in MQNLI_FILES if not (TUTORIAL_DATA / fn).exists()]
for fn in missing:
    subprocess.run(["curl", "-fsSL", BASE + fn, "-o", str(TUTORIAL_DATA / fn)], check=True)
still_missing = [fn for fn in MQNLI_FILES if not (TUTORIAL_DATA / fn).exists()]
if still_missing:
    raise FileNotFoundError(f"Missing MQNLI signature files: {still_missing}")

print("Repo root:", Path.cwd())
print("MQNLI signature files ready:", len(MQNLI_FILES))
print("If packages were newly installed, restart the runtime and re-run from this cell.")


In [ ]:
# Optional: install nnsight only if you plan to add NNsight experiments.
# The core notebook below uses transformers + pyvene and does not require nnsight.
RUN_NNSIGHT_INSTALL = False

if RUN_NNSIGHT_INSTALL:
    import sys, subprocess, importlib.util
    if importlib.util.find_spec("nnsight") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nnsight"])
    else:
        print("nnsight already installed")
else:
    print("Skipping optional nnsight install.")


In [ ]:
# ── Library imports + device/seed ──────────────────────────────────────────
import json, gc, copy
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

import nli_das
from nli_das import (
    LexicalCausalModel,
    LEXICAL_PAIRS, UPWARD_TEMPLATES, DOWNWARD_TEMPLATES, DEFAULT_TEMPLATES,
    NLIExample, generate_examples, label_distribution, relation_distribution,
    build_counterfactual_dataset, build_random_source_dataset,
    build_wrong_variable_dataset, pair_level_split,
    LabelVerbalizer, compute_iia,
    make_das_config, train_das_alignment, evaluate_das_iia,
    run_patching_sweep, save_patching_heatmap_from_df,
)

SEED   = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"DEVICE = {DEVICE}")


---
## 1. Why DAS?

### The localization problem

A central goal of mechanistic interpretability is to **localize** an abstract concept to a component of a deep model. *Where* in GPT-2 does the model represent "the lexical relation between two words"? The naive approach is to **patch** the entire residual-stream activation at some `(layer, position)` from a source input into a base input, and check whether the model's output changes the way a symbolic causal model predicts. This is **activation patching** (also called interchange intervention).

But residual-stream vectors are **polysemantic** — a single vector encodes many features simultaneously. If we patch the full vector at the hypothesis-word position, we change *everything* encoded there: lexical category, semantic role, position, frequency, syntactic dependencies. We get behavioral effects but not a clean mechanism.

### What DAS adds

**DAS** (Geiger et al. 2023) learns an *orthogonal rotation* $R$ of the residual stream so that we can patch a **low-dimensional subspace** rather than the full vector. The rotation is trained — by gradient descent on counterfactual data — so that patching the subspace transfers the *target high-level variable* (e.g. `lexical_relation`) from source to base while leaving everything else alone.

Concretely, at a chosen `(layer, position, component)` site:

1. Forward-pass the base prompt and the source prompt.
2. Take the activation $h$ at the site for each.
3. Rotate into feature space: $f = h R^\top$.
4. **Swap the first $k$ features** of base with those of source.
5. Rotate back: $h' = f R$.
6. Continue the forward pass.
7. Loss: cross-entropy on the predicted label against the *symbolic counterfactual label* (what the high-level causal model predicts under the same intervention).

Only $R$ is trained — the LM is frozen. If a $k$-dim subspace can be found that aligns with the target variable, the loss drops; if not, the loss plateaus near chance.

### The metric: Interchange Intervention Accuracy (IIA)

For a held-out batch, IIA is the fraction of examples where the *patched* model's argmax label equals the *symbolic counterfactual* label. IIA = 1 means perfect causal abstraction at the chosen site; IIA at chance means the rotation found nothing meaningful.

We compare every result against three baselines:

- **Chance** (3-way label ≈ 0.33)
- **Random-source control**: same DAS rotation, but each base is paired with a *random* source. If the rotation actually transports the target variable from source to base, random sources should give random label-changes and IIA collapses to chance.
- **Wrong-variable control**: same DAS rotation evaluated on counterfactuals computed for a *different* target variable. If the rotation specifically encoded `lexical_relation`, evaluating it as if it encoded `premise_word_identity` gives low IIA.

These controls together establish that the DAS subspace really aligns with the target variable, not just with any source-dependent direction.

---
## 2. The high-level causal model

Before we look at any neural network, we write down the symbolic algorithm we *hope* GPT-2 has learned for our lexical NLI task. The model takes a premise word and a hypothesis word, computes their lexical relation, and outputs an NLI label.

```
inputs        :  premise_word, hypothesis_word, context (= template)
intermediate  :  lexical_relation ∈ {EQUIV, FORWARD, REVERSE, DISJOINT}
                 monotonicity     ∈ {upward, downward}  (from context)
output        :  label ∈ {entailment, neutral, contradiction}
```

The relation → label table depends on monotonicity:

|             | upward (e.g. "A X is on the table")       | downward (e.g. "No X is on the table") |
|---|---|---|
| `EQUIV`     | entailment                                 | entailment                              |
| `FORWARD`   | entailment   (hyponym→hypernym preserves)  | neutral      (negation flips)           |
| `REVERSE`   | neutral      (hypernym→hyponym loses info) | entailment   (flips back)               |
| `DISJOINT`  | contradiction                              | contradiction                           |

The `LexicalCausalModel` class in `nli_das` materializes this table and exposes `.run(base, interventions=...)` so we can ask *"what label would the symbolic model emit if we intervened on `lexical_relation = FORWARD`?"* — this is what gives us **gold counterfactual labels** for training and evaluating DAS.

In [ ]:
# ── Inspect the causal model ───────────────────────────────────────────────
causal = LexicalCausalModel(monotonicity="upward")

# Forward pass on (premise_word=dog, hypothesis_word=animal) — should give ENTAILMENT
base = causal.run(premise_word="dog", hypothesis_word="animal", context="on_the_table")
print(f"Base trace:  rel={base['lexical_relation']}, label={base['label']}")

# Counterfactual: pretend the lexical relation is DISJOINT instead
cf = causal.run(premise_word="dog", hypothesis_word="animal", context="on_the_table",
                interventions={"lexical_relation": "DISJOINT"})
print(f"After do(lexical_relation=DISJOINT):  label={cf['label']}")

# Monotonicity intervention: same words, but downward context
cf_mono = causal.run(premise_word="dog", hypothesis_word="animal", context="no_x_on_table",
                    interventions={"monotonicity": "downward"})
print(f"After do(monotonicity=downward):  rel={cf_mono['lexical_relation']}, label={cf_mono['label']}")


---
## 3. Generating controlled NLI examples

We need a dataset where:

- the lexical relation between every (premise, hypothesis) pair is **symbolically known** (so we can compute counterfactual labels for free),
- the **token position** of the content word is deterministic per template (so DAS can intervene at a fixed site without padding artifacts),
- the four relation classes are **roughly balanced** (so IIA isn't dominated by one class).

We auto-generate pairs from a small hand-curated hypernym ontology (about 50 leaf words across categories like *mammal*, *bird*, *vehicle*, *tool*, *furniture*, *fruit*). The four relations are derived from the closure of the ontology, with `DISJOINT` pairs subsampled per-word so no leaf dominates.

In [ ]:
# ── Vocabulary statistics ─────────────────────────────────────────────────
from nli_das.data import auto_generate_pairs
print(f"Lexical pairs: {len(LEXICAL_PAIRS)}")
print("Relation distribution:", dict(Counter(r for _, _, r in LEXICAL_PAIRS)))

# Template inventory
print(f"\nUpward templates  ({len(UPWARD_TEMPLATES)}):")
for t in UPWARD_TEMPLATES:
    print(f"  {t.name:18s}  {t.premise_format!r}")
print(f"\nDownward templates ({len(DOWNWARD_TEMPLATES)}):")
for t in DOWNWARD_TEMPLATES:
    print(f"  {t.name:18s}  {t.premise_format!r}")

# Materialise all (pair × template) examples
examples = generate_examples()
print(f"\nTotal examples: {len(examples)}")
print("Label distribution:", label_distribution(examples))
print("\nSample prompts:")
for ex in examples[:5]:
    print(f"  [{ex.label:14s}]  {ex.prompt!r}")


---
## 4. Counterfactual pairs + the two controls

For DAS we don't just need single examples — we need **(base, source) pairs** plus a gold counterfactual label that says: "if you patched `lexical_relation` from source into base, the model *should* output this label". The function `build_counterfactual_dataset` materializes these tuples up front, including the **exact token position** of the intervention site (we localize the content word's first BPE token).

Three more pieces of hygiene that make this evaluation robust:

1. **Pair-level holdout.** We split the lexical pairs into train/eval *before* sampling counterfactuals, so eval lexical items are never seen at training time. This rules out "the rotation memorized the (premise, hypothesis) prompts."
2. **`require_label_change=True`.** We discard pairs where the counterfactual label equals the base label — otherwise IIA gets inflated by trivially-stuck "predict the base label" behavior. This is the DAS-paper convention.
3. **Two controls, built from the same eval set:**
   - **Random-source**: for each base, replace the source with a randomly-chosen source from the pool and recompute the gold counterfactual label. If DAS really transports the source's `lexical_relation` value, this control's IIA should drop to chance.
   - **Wrong-variable**: same (base, source) pairs and intervention site, but recompute the gold counterfactual label as if the target variable were `premise_word_identity` instead of `lexical_relation`. If the learned subspace specifically encodes `lexical_relation`, it should score poorly here.

In [ ]:
# ── Build train + eval datasets and the two controls ───────────────────────
TARGET_VAR = "lexical_relation"
N_TRAIN, N_EVAL = 512, 128

tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
verbalizer = LabelVerbalizer.from_tokenizer(
    tokenizer,
    {"entailment": " yes", "neutral": " maybe", "contradiction": " no"},
)
print(f"verbalizer token ids = {verbalizer.token_ids}")

# Hold out 20% of lexical pairs for evaluation.
train_pairs, eval_pairs = pair_level_split(LEXICAL_PAIRS, train_frac=0.8, seed=SEED)
print(f"\nPair-level split: train={len(train_pairs)} pairs, eval={len(eval_pairs)} pairs")

# Pick one template for the lexical-relation sections so positions align.
TEMPLATES = [UPWARD_TEMPLATES[0]]   # "A {word} is on the table."

train_ds = build_counterfactual_dataset(
    tokenizer, target_variable=TARGET_VAR,
    pairs=train_pairs, templates=TEMPLATES,
    n_examples=N_TRAIN, seed=SEED, require_label_change=True,
)
eval_ds = build_counterfactual_dataset(
    tokenizer, target_variable=TARGET_VAR,
    pairs=eval_pairs, templates=TEMPLATES,
    n_examples=N_EVAL, seed=SEED + 42, require_label_change=True,
)
print(f"\nCounterfactual datasets:  train={len(train_ds)}  eval={len(eval_ds)}")

# Build the two controls from the eval set.
eval_random = build_random_source_dataset(eval_ds, seed=SEED + 1)
eval_wrong  = build_wrong_variable_dataset(eval_ds, wrong_variable="premise_word_identity")
print(f"\nControls:  random-source n={len(eval_random)}  wrong-variable n={len(eval_wrong)}")

# Show one concrete (base, source, cf_label) tuple
ex = train_ds.examples[0]
print(f"\nExample CF pair:")
print(f"  base    : {ex.base.prompt!r}   →  base_label={ex.base.label}")
print(f"  source  : {ex.source.prompt!r}   →  source_label={ex.source.label}")
print(f"  patch   : do({ex.target_variable}={ex.source.lexical_relation!r}) at token pos {ex.intervention_pos}")
print(f"  cf_label: {['entailment','neutral','contradiction'][ex.counterfactual_label_id]}")


---
## 5. Fine-tuning GPT-2 on the lexical NLI task

**Why this step is mandatory.** DAS learns an orthogonal rotation so that patching a low-dimensional subspace
transports a high-level variable from source to base. If the base model does not solve the factual task, the
intervention target is not meaningful.

We fine-tune GPT-2 on factual NLI examples using label-token loss. Unlike the earlier draft, this audited version
trains on one upward and one downward template. That keeps the lexical-relation section fixed to the upward template,
while giving the later monotonicity section a real chance: the model has seen that `A ...` and `No ...` induce
different relation-to-label maps.


In [ ]:
# ── Fine-tune GPT-2 on factual lexical NLI ──────────────────────────────────
# Key fixes vs original: (1) loss on label token only → clean gradient signal,
# (2) 20 epochs at 2e-4 LR → sufficient convergence on ~336 examples.

from nli_das.data import generate_examples

LABEL_STR     = {"entailment": " yes", "neutral": " maybe", "contradiction": " no"}
NUM_FT_EPOCHS = 20
FT_LR         = 2e-4
FT_BSZ        = 16

FT_TEMPLATES = [UPWARD_TEMPLATES[0], DOWNWARD_TEMPLATES[0]]
factual_train = generate_examples(pairs=train_pairs, templates=FT_TEMPLATES)
factual_eval  = generate_examples(pairs=eval_pairs,  templates=FT_TEMPLATES)
print(f"Fine-tune templates: {[t.name for t in FT_TEMPLATES]}")
print(f"Fine-tune:  {len(factual_train)} train  |  {len(factual_eval)} eval")
print("Label dist (train):", {k: sum(1 for e in factual_train if e.label == k)
                               for k in ("entailment", "neutral", "contradiction")})

def _encode_label_only(exs, tok, lmap):
    """Encode prompt + label token; return list of (prompt_ids, label_token_id)."""
    out = []
    for ex in exs:
        pids = tok.encode(ex.prompt, add_special_tokens=False)
        lids = tok.encode(lmap[ex.label], add_special_tokens=False)
        assert len(lids) == 1, f"label {ex.label!r} → {lids} (not single-token)"
        out.append((pids, lids[0]))
    return out

def _collate_label(batch):
    """Pad to longest; loss ONLY on the label token (last position)."""
    maxlen = max(len(p) + 1 for p, _ in batch)
    n = len(batch)
    input_ids = torch.zeros(n, maxlen, dtype=torch.long)
    attn_mask  = torch.zeros(n, maxlen, dtype=torch.long)
    labels     = torch.full((n, maxlen), -100, dtype=torch.long)
    for i, (pids, lid) in enumerate(batch):
        t = len(pids)
        input_ids[i, :t] = torch.tensor(pids)
        input_ids[i, t]  = lid
        attn_mask[i, :t + 1] = 1
        labels[i, t]     = lid   # CE only on this one token
    return input_ids, attn_mask, labels

train_seqs = _encode_label_only(factual_train, tokenizer, LABEL_STR)
eval_seqs  = _encode_label_only(factual_eval,  tokenizer, LABEL_STR)

# ── Training loop ────────────────────────────────────────────────────────────
ft_model  = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
optimizer = torch.optim.AdamW(ft_model.parameters(), lr=FT_LR, weight_decay=0.01)
rng_ft    = np.random.default_rng(SEED)

print(f"\nTraining {NUM_FT_EPOCHS} epochs (label-token loss only) …")
epoch_losses = []
ft_model.train()
for epoch in range(NUM_FT_EPOCHS):
    order = rng_ft.permutation(len(train_seqs)).tolist()
    tot_loss = 0; n_b = 0
    for start in range(0, len(order), FT_BSZ):
        batch = [train_seqs[i] for i in order[start : start + FT_BSZ]]
        ids, mask, lbl = _collate_label(batch)
        ids, mask, lbl = ids.to(DEVICE), mask.to(DEVICE), lbl.to(DEVICE)
        out = ft_model(input_ids=ids, attention_mask=mask, labels=lbl)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        optimizer.step(); optimizer.zero_grad()
        tot_loss += out.loss.item(); n_b += 1
    epoch_losses.append(tot_loss / n_b)
    if (epoch + 1) % 5 == 0:
        print(f"  epoch {epoch+1}/{NUM_FT_EPOCHS}  loss={epoch_losses[-1]:.4f}")

ft_model.eval()

# ── Training-loss curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(range(1, NUM_FT_EPOCHS + 1), epoch_losses, marker="o", color="tab:blue")
ax.set_xlabel("Epoch"); ax.set_ylabel("Label-token CE loss")
ax.set_title("GPT-2 fine-tuning on lexical NLI (label-only loss)"); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/figures/finetuning_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Factual accuracy ──────────────────────────────────────────────────────────
verb_tids = verbalizer.token_id_tensor(device=DEVICE)   # [3] LongTensor
correct = total = 0
with torch.no_grad():
    for ex, (pids, _) in zip(factual_eval, eval_seqs):
        prompt_ids  = torch.tensor([pids]).to(DEVICE)
        logits_next = ft_model(input_ids=prompt_ids).logits[0, -1]   # [V]
        verb_logits = logits_next[verb_tids]                          # [3]
        pred        = verb_logits.argmax().item()
        correct    += int(pred == ex.label_id)
        total      += 1

factual_acc = correct / total
print(f"\nFactual accuracy (fine-tuned GPT-2, eval): {factual_acc:.1%}  ({correct}/{total})")
if factual_acc < 0.55:
    print("WARNING: accuracy still low — run more epochs or raise FT_LR.")
else:
    print("Sufficient factual accuracy. Proceeding.")

# Freeze for DAS — only the rotation gets gradient updates downstream.
model = ft_model
for p in model.parameters():
    p.requires_grad_(False)
n_layers = model.config.n_layer

del optimizer, train_seqs, eval_seqs, ft_model
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print(f"Model frozen: gpt2  ({n_layers} layers, hidden={model.config.n_embd})")

---
## 6. Activation patching baseline

Before training DAS, we run **vanilla activation patching** as a coarse baseline. For every `(layer, component, position)` cell in a grid, we patch the source's activation into the base's run, then measure two things:

- **Logit-difference recovery**: how much of the (clean − corrupted) logit gap the patch closes (after Wang et al. 2022, *Interpretability in the Wild*).
- **IIA**: argmax accuracy of the patched run against the symbolic counterfactual label.

Patching does *not* learn anything — it just swaps full activation vectors. So patching gets to "high cause" but can never separate the target variable from everything else encoded in the same vector. The heatmap tells us *where* in the network the relevant information is most concentrated, which gives DAS a sensible site to anchor the rotation.

In [ ]:
# ── Patching sweep ─────────────────────────────────────────────────────────
# Layer × component × position, on a 64-example subset (~3 min on T4).
# `model` is the fine-tuned, frozen GPT-2 from §5.
print(f"Fine-tuned GPT-2: {n_layers} layers, hidden={model.config.n_embd}")

# Smaller dataset for the patching sweep — the cell will run 12 layers × 3 components × seq_len positions.
patch_ds = build_counterfactual_dataset(
    tokenizer, target_variable=TARGET_VAR,
    pairs=train_pairs, templates=TEMPLATES,
    n_examples=64, seed=SEED, require_label_change=True,
)
print(f"Patching dataset: {len(patch_ds)}")

patch_df = run_patching_sweep(
    model, tokenizer, patch_ds.examples,
    layers=list(range(n_layers)),
    components=("mlp_output", "attention_input", "block_output"),
    positions="all",
    metric="logit_recovery",
    device=DEVICE, verbalizer=verbalizer,
    batch_size=8, progress=True,
)
patch_df.to_csv("outputs/tables/patching_sweep.csv", index=False)
print(f"\nSweep rows: {len(patch_df)}  base_accuracy={patch_df.attrs.get('base_accuracy', 0):.3f}")


In [ ]:
# ── Heatmaps + best-site selection ─────────────────────────────────────────
for comp in ("mlp_output", "attention_input", "block_output"):
    save_patching_heatmap_from_df(
        patch_df, f"outputs/figures/patching_{comp}.png",
        value_col="iia_correct", component=comp, annot=True,
    )

# Best (layer, component, position) by mean IIA then logit recovery.
site_scores = (
    patch_df.groupby(["component", "layer", "position"])
    .agg(iia=("iia_correct", "mean"), recovery=("recovery", "mean"))
    .reset_index()
    .sort_values(["iia", "recovery"], ascending=False)
)
print("Top 5 patching sites (all positions):")
print(site_scores.head().to_string(index=False))

# ── Restrict to positions DAS can actually use ───────────────────────────────
# Label-only fine-tuning often makes the final Answer: token the best full-vector
# patching site. That is real, but the CF dataset's intervention positions are
# content-word positions. If we selected the Answer: token and then silently fell
# back to a content-word position later, the layer/position pair would be incoherent.
content_positions = sorted({int(ex.intervention_pos) for ex in train_ds.examples})
valid_sites = site_scores[site_scores["position"].isin(content_positions)].copy()
assert len(valid_sites) > 0, "No patching sites match intervention positions in train_ds."

print(f"\nContent-word positions in CF dataset: {content_positions}")
print("Top 5 patching sites (content-word positions only):")
print(valid_sites.head().to_string(index=False))

best = valid_sites.iloc[0]
COMPONENT      = str(best["component"])
BEST_LAYER     = int(best["layer"])
FIXED_POSITION = int(best["position"])
print(f"\nSelected DAS anchor:  layer={BEST_LAYER}  component={COMPONENT}  position={FIXED_POSITION}")
print(f"  patching IIA at this usable site: {float(best['iia']):.3f}")


**Interpreting the patching result.** The heatmap should show that *full-vector* patching at the best site achieves moderate IIA — significantly above chance, but well short of perfect. This is exactly the limitation DAS is designed to address: the full activation carries the target variable *plus* a lot of other stuff, so the patch transports too much.

---
## 7. Single-variable DAS — `lexical_relation`

Now we train the DAS rotation. The intervenable model wraps GPT-2 with a `LowRankRotatedSpaceIntervention` at the (layer, component, position) site picked by patching. The only trainable parameters are the orthogonal rotation matrix; the LM stays frozen.

We filter the train and eval datasets to the chosen `FIXED_POSITION` so every batch patches the same token index (pyvene's rotation layer can't handle variable positions cleanly).

**Hyperparameters** chosen from preliminary sweeps:
- Subspace dimension `d=16` (the symbolic variable has 4 values but the neural encoding can be more distributed)
- 20 epochs, batch size 8, Adam lr 1e-3
- We also pass `eval_cf_dataset=eval_ds` so the training loop logs val loss + val IIA each epoch (catches overfitting on small data).

In [ ]:
from collections import Counter

# ── Filter lexical datasets to the selected fixed intervention position ──────
# Important: do NOT silently change FIXED_POSITION here. The layer was chosen for
# this exact position in the patching sweep. If this assertion fails, rerun §6–§7.
assert any(ex.intervention_pos == FIXED_POSITION for ex in train_ds.examples), (
    f"Selected FIXED_POSITION={FIXED_POSITION} is absent from train_ds; "
    "rerun the patching-site selection cell."
)

train_ds_f      = train_ds.filter_by_position(FIXED_POSITION)
eval_ds_f       = eval_ds.filter_by_position(FIXED_POSITION)
eval_random_f   = eval_random.filter_by_position(FIXED_POSITION)
eval_wrong_f    = eval_wrong.filter_by_position(FIXED_POSITION)

print(f"After filter to position {FIXED_POSITION}:")
print(f"  train={len(train_ds_f)}  eval={len(eval_ds_f)}  random={len(eval_random_f)}  wrong={len(eval_wrong_f)}")
print("  original train position counts:", dict(Counter(ex.intervention_pos for ex in train_ds.examples)))
print("  filtered train position counts:", dict(Counter(ex.intervention_pos for ex in train_ds_f.examples)))

assert len(train_ds_f) >= 64, "Too few lexical training CF examples after position filter."
assert len(eval_ds_f)  >= 32, "Too few lexical eval CF examples after position filter."
assert len(eval_random_f) == len(eval_ds_f), "Random-source control size mismatch after filtering."
assert len(eval_wrong_f)  == len(eval_ds_f), "Wrong-variable control size mismatch after filtering."


In [ ]:
# ── Train DAS ──────────────────────────────────────────────────────────────
DIM        = 16
NUM_EPOCHS = 20

torch.manual_seed(SEED)
cfg = make_das_config(model, layer=BEST_LAYER, component=COMPONENT,
                      intervention_dim=DIM, unit="pos")
das_out = train_das_alignment(
    model, tokenizer, train_ds_f, cfg,
    num_epochs=NUM_EPOCHS, lr=1e-3, batch_size=8,
    device=DEVICE, verbalizer=verbalizer,
    fixed_position=FIXED_POSITION,
    eval_cf_dataset=eval_ds_f,    # logs val loss + val IIA every epoch
    progress=True,
)

# Training curve
hist = pd.DataFrame(das_out.history)
print("\nLast 3 epochs:")
print(hist.tail(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist["epoch"], hist["loss"], marker="o", label="train loss")
if "val_loss" in hist.columns: ax[0].plot(hist["epoch"], hist["val_loss"], marker="s", label="val loss")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hist["epoch"], hist["train_iia"], marker="o", label="train IIA")
if "val_iia" in hist.columns: ax[1].plot(hist["epoch"], hist["val_iia"], marker="s", label="val IIA")
ax[1].axhline(1/3, color="black", linestyle="--", lw=1, label="chance")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("IIA"); ax[1].set_ylim(0, 1.05); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.suptitle(f"DAS training — L{BEST_LAYER} {COMPONENT} pos{FIXED_POSITION} d{DIM}")
fig.tight_layout(); fig.savefig("outputs/figures/lexical_das_training.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Evaluate against controls ────────────────────────────────────────────────
res        = evaluate_das_iia(das_out.intervenable, eval_ds_f, tokenizer,
                              device=DEVICE, verbalizer=verbalizer, fixed_position=FIXED_POSITION)
res_random = evaluate_das_iia(das_out.intervenable, eval_random_f, tokenizer,
                              device=DEVICE, verbalizer=verbalizer, fixed_position=FIXED_POSITION)
res_wrong  = evaluate_das_iia(das_out.intervenable, eval_wrong_f,  tokenizer,
                              device=DEVICE, verbalizer=verbalizer, fixed_position=FIXED_POSITION)

summary = pd.DataFrame([
    {"condition": "Trained DAS",     "IIA": res["iia"]},
    {"condition": "Random-source",   "IIA": res_random["iia"]},
    {"condition": "Wrong-variable",  "IIA": res_wrong["iia"]},
]).round(3)
print(summary.to_string(index=False))
summary.to_csv("outputs/tables/lexical_relation_summary.csv", index=False)
print(f"\nFactual accuracy (base prediction, no intervention): {res['factual_accuracy']:.3f}")
print(f"Verbalizer-hit rate (unrestricted top-token is yes/no/maybe): {res['verbalizer_hit_rate']:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["tab:blue", "tab:gray", "tab:orange"]
ax.bar(summary["condition"], summary["IIA"], color=colors)
ax.axhline(1/3, color="black", linestyle="--", lw=1, label="3-way chance")
ax.set_ylabel("IIA"); ax.set_ylim(0, 1.05)
ax.set_title(f"DAS on lexical_relation — L{BEST_LAYER} {COMPONENT} d{DIM}")
ax.legend(); fig.tight_layout()
fig.savefig("outputs/figures/lexical_relation_bars.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Random-source sanity check ───────────────────────────────────────────────
# Random-source is a generalization check, not the main negative control: its CF
# labels are recomputed from the random source, so high IIA can be good. The
# wrong-variable condition is the cleaner negative control.
from collections import Counter as _C21
same_cf = sum(
    int(a.counterfactual_label_id == b.counterfactual_label_id)
    for a, b in zip(eval_ds_f.examples, eval_random_f.examples)
)
print(f"\nRandom-source sanity:")
print(f"  Trained CF labels:       {dict(_C21(int(e.counterfactual_label_id) for e in eval_ds_f.examples))}")
print(f"  Random-source CF labels: {dict(_C21(int(e.counterfactual_label_id) for e in eval_random_f.examples))}")
print(f"  Same CF label (trained vs random): {same_cf}/{len(eval_ds_f)} ({same_cf/len(eval_ds_f):.1%})")
print("  Interpretation: random-source tests whether the learned rotation generalizes")
print("  to arbitrary valid sources. Wrong-variable is the stricter negative control.")


**Interpreting the result.**

The important comparison is:

- **Trained DAS IIA**: does the learned subspace reproduce the symbolic counterfactual labels?
- **Wrong-variable IIA**: the main negative control. We score the same intervention against labels for a different
  high-level variable; this should be low.
- **Random-source IIA**: a generalization check, not a negative control. The dataset recomputes the gold
  counterfactual label for the new random source, so high random-source IIA means the rotation can transport
  arbitrary valid source values, which is good. It should not be described as "chance-level expected."

The actual evidence is high trained/random-source generalization, low wrong-variable IIA, and low random-init model
calibration later in the notebook.


---
## 8. A second variable — `monotonicity`

`monotonicity` is the upward/downward feature supplied by the template. This section is now an actual attempted
repair rather than an expected failure: the factual model was trained above on both an upward and a downward
template, so it should have learned that `A` versus `No` changes how lexical relations map to labels.

Two differences from the lexical-relation setup:

1. **Different intervention position.** We patch the monotonicity marker token (`A` or `No`), usually position 0.
2. **Layer selected by patching.** We do not hardcode an "early syntactic" layer; we run a small full-vector
   patching sweep at the monotonicity marker and choose the best layer before DAS.

If this still fails, it is an interpretable negative result: the fine-tuned GPT-2 did not internalize the
monotonicity variable in a clean local subspace at this site.


In [ ]:
# ── Build the monotonicity counterfactual dataset ──────────────────────────
UP_T   = [UPWARD_TEMPLATES[0]]    # "A {word} is on the table."
DOWN_T = [DOWNWARD_TEMPLATES[0]]  # "No {word} is on the table."

mono_train = build_counterfactual_dataset(
    tokenizer, target_variable="monotonicity",
    pairs=train_pairs, templates=UP_T, source_templates=DOWN_T,
    n_examples=256, seed=SEED, require_label_change=True,
    require_monotonicity_flip=True,
)
mono_eval = build_counterfactual_dataset(
    tokenizer, target_variable="monotonicity",
    pairs=eval_pairs, templates=UP_T, source_templates=DOWN_T,
    n_examples=64, seed=SEED + 42, require_label_change=True,
    require_monotonicity_flip=True,
)

# Marker token position
MONO_POS = Counter(int(ex.intervention_pos) for ex in mono_train.examples).most_common(1)[0][0]
print(f"Monotonicity-marker token position: {MONO_POS}  (expect 0 — the determiner)")

mono_train = mono_train.filter_by_position(MONO_POS)
mono_eval  = mono_eval.filter_by_position(MONO_POS)
mono_eval_random = build_random_source_dataset(mono_eval, seed=SEED + 1)
mono_eval_wrong  = build_wrong_variable_dataset(mono_eval, wrong_variable="hypothesis_word_identity")
print(f"After filter:  train={len(mono_train)}  eval={len(mono_eval)}")
print(f"\nExample monotonicity pair:")
ex = mono_train.examples[0]
print(f"  base   : {ex.base.prompt!r}")
print(f"  source : {ex.source.prompt!r}")
print(f"  patch  : token pos {ex.intervention_pos}  (the determiner)")
print(f"  cf_label: {['entailment','neutral','contradiction'][ex.counterfactual_label_id]}")


In [ ]:
# Train DAS on monotonicity after selecting a layer by full-vector patching.
MONO_DIM = 16

mono_patch_df = run_patching_sweep(
    model, tokenizer, mono_train.examples[: min(64, len(mono_train.examples))],
    layers=list(range(n_layers)),
    components=("block_output",),
    positions=[MONO_POS],
    metric="logit_recovery",
    device=DEVICE, verbalizer=verbalizer,
    batch_size=8, progress=True,
)
mono_scores = (
    mono_patch_df.groupby(["component", "layer", "position"])
    .agg(iia=("iia_correct", "mean"), recovery=("recovery", "mean"))
    .reset_index()
    .sort_values(["iia", "recovery"], ascending=False)
)
print("Top monotonicity patching sites:")
print(mono_scores.head(5).to_string(index=False))
MONO_LAYER = int(mono_scores.iloc[0]["layer"])
MONO_COMPONENT = str(mono_scores.iloc[0]["component"])
print(f"Selected monotonicity DAS anchor: L{MONO_LAYER} {MONO_COMPONENT} pos{MONO_POS}")

torch.manual_seed(SEED)
mono_cfg = make_das_config(model, layer=MONO_LAYER, component=MONO_COMPONENT,
                           intervention_dim=MONO_DIM, unit="pos")
mono_out = train_das_alignment(
    model, tokenizer, mono_train, mono_cfg,
    num_epochs=20, lr=1e-3, batch_size=8,
    device=DEVICE, verbalizer=verbalizer,
    fixed_position=MONO_POS,
    eval_cf_dataset=mono_eval,
    progress=True,
)

m_res   = evaluate_das_iia(mono_out.intervenable, mono_eval,        tokenizer,
                           device=DEVICE, verbalizer=verbalizer, fixed_position=MONO_POS)
m_res_r = evaluate_das_iia(mono_out.intervenable, mono_eval_random, tokenizer,
                           device=DEVICE, verbalizer=verbalizer, fixed_position=MONO_POS)
m_res_w = evaluate_das_iia(mono_out.intervenable, mono_eval_wrong,  tokenizer,
                           device=DEVICE, verbalizer=verbalizer, fixed_position=MONO_POS)

mono_summary = pd.DataFrame([
    {"condition": "Trained DAS", "IIA": m_res["iia"]},
    {"condition": "Random-source generalization", "IIA": m_res_r["iia"]},
    {"condition": "Wrong-variable control", "IIA": m_res_w["iia"]},
]).round(3)
print(mono_summary.to_string(index=False))
mono_summary.to_csv("outputs/tables/monotonicity_summary.csv", index=False)

if m_res["iia"] <= max(1/3, m_res_w["iia"] + 0.05):
    print("\nWARNING: monotonicity DAS does not clearly beat the control/baseline.")
    print("Interpret this as a negative or preliminary result, not as a clean second-variable success.")
else:
    print("\nMonotonicity DAS beats the wrong-variable control; this supports a second-variable alignment.")


In [ ]:
# ── Side-by-side: lexical_relation vs monotonicity ─────────────────────────
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(3); width = 0.36
a_vals = [res["iia"], res_random["iia"], res_wrong["iia"]]
e_vals = [m_res["iia"], m_res_r["iia"], m_res_w["iia"]]
ax.bar(x - width/2, a_vals, width, color="tab:blue",
       label=f"lexical_relation  (L{BEST_LAYER}, d{DIM}, pos{FIXED_POSITION})")
ax.bar(x + width/2, e_vals, width, color="tab:purple",
       label=f"monotonicity     (L{MONO_LAYER}, d{MONO_DIM}, pos{MONO_POS})")
ax.set_xticks(x); ax.set_xticklabels(["Trained", "Random-source\ngeneralization", "Wrong-variable\ncontrol"])
ax.axhline(1/3, color="black", linestyle="--", lw=1, label="3-way chance")
ax.set_ylabel("IIA"); ax.set_ylim(0, 1.05)
ax.set_title("Lexical relation and monotonicity DAS summaries")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout(); fig.savefig("outputs/figures/two_variables.png", dpi=150, bbox_inches="tight")
plt.show()


**Why this matters.**

A successful monotonicity result would show that the same frozen GPT-2 supports two different high-level variables
at two different intervention sites. A failed monotonicity result is still useful: it shows that fine-tuning on the
factual labels is not automatically enough to make every symbolic variable locally recoverable by DAS.

For the final presentation, treat the lexical-relation section as the clean result and monotonicity as either a
repaired second-variable result or a carefully controlled negative result, depending on the run.


---
## 9. MQNLI — the nested causal graph (faithful pyvene replication)

Sections 9–13 follow the **official pyvene MQNLI tutorial** as closely as possible
(`stanfordnlp/pyvene` advanced tutorials, adapted by Amir Zur). The earlier hand-rolled
MQNLI experiment in previous drafts failed for two structural reasons, and this rewrite
fixes both by adopting the tutorial's method wholesale:

1. **Intervene at a single fixed "decision" position, not at a content token.**
   We tokenize every example to a fixed length and read the answer off position
   `MAX_LENGTH-2`. Because GPT-2 is autoregressive, that late position has attended to the
   *entire* premise and hypothesis, so any high-level MQNLI variable is in principle
   recoverable there. Patching a premise token (the old approach) is causally impossible
   for variables whose inputs appear later in the string.
2. **Use pyvene's `CausalModel` + `generate_counterfactual_dataset`.** The counterfactual
   source is *sampled to realize a specific value* of the target high-level variable, and
   the gold label is computed by `run_interchange`. This guarantees clean single-variable
   interventions instead of the non-minimal pairs that broke the old version.

The MQNLI graph has **33 variables**. Leaf nodes are words; interior nodes are Natural-Logic
relations (`independence, equivalence, entails, reverse entails, contradiction, alternation, cover`).
The root `QP_S` is the sentence-level relation the model must predict.

```
        leaves (words)            relations                 root
  Q/N/Adj/V/Adv/Neg (P & H)  →  N_O, Adj_O, V, Adv ...  →
                               →  NP_O, VP, Q_O  →  QP_O  →
                               →  Neg  →  NegP  ─────────────┐
                               →  NP_S, Q_S  ────────────────┴→ QP_S
```

The model is **GPT-2 trained from scratch-ish on MQNLI** (via pyvene's `create_gpt2_lm`) to
predict the relation word after `Relation:`. This is a *different* model from the lexical
GPT-2 in §5 — the MQNLI task has its own vocabulary of answer tokens (the 7 relations).


In [ ]:
# ── Section 9: build the MQNLI CausalModel exactly as in the pyvene tutorial ──
import json, random, copy, itertools, re, os
import numpy as np, pandas as pd
import torch
from collections import Counter
from pyvene import CausalModel

# The 5 Natural-Logic signature tables were downloaded by the bootstrap cell.
class Hashabledict(dict):
    def __hash__(self): return hash(frozenset(self))

_TD = "tutorial_data"
with open(f"{_TD}/mqnli_q_projectivity.json") as f:
    determiner_signatures = json.load(f)
    determiner_signatures = Hashabledict({
        q1: Hashabledict({q2: Hashabledict({r1: Hashabledict(determiner_signatures[q1][q2][r1])
            for r1 in determiner_signatures[q1][q2]}) for q2 in determiner_signatures[q1]})
        for q1 in determiner_signatures})
with open(f"{_TD}/mqnli_neg_signature.json") as f:      negation_signature = Hashabledict(json.load(f))
with open(f"{_TD}/mqnli_empty_signature.json") as f:    emptystring_signature = Hashabledict(json.load(f))
with open(f"{_TD}/mqnli_cont_signature.json") as f:     compose_contradiction_signature = Hashabledict(json.load(f))
with open(f"{_TD}/mqnli_neg_cont_signature.json") as f: compose_neg_contradiction_signature = Hashabledict(json.load(f))

EMPTY=""; IND="independence"; EQV="equivalence"; ENT="entails"
REV="reverse entails"; CON="contradiction"; ALT="alternation"; COV="cover"
RELATIONS=[IND,EQV,ENT,REV,CON,ALT,COV]

parents = {
    "N_P_O":[], "N_H_O":[], "N_O":["N_P_O","N_H_O"],
    "Adj_P_O":[], "Adj_H_O":[], "Adj_O":["Adj_P_O","Adj_H_O"],
    "NP_O":["Adj_O","N_O"],
    "Q_P_O":[], "Q_H_O":[], "Q_O":["Q_P_O","Q_H_O"],
    "V_P":[], "V_H":[], "V":["V_P","V_H"],
    "Adv_P":[], "Adv_H":[], "Adv":["Adv_P","Adv_H"],
    "VP":["Adv","V"],
    "QP_O":["Q_O","NP_O","VP"],
    "Neg_P":[], "Neg_H":[], "Neg":["Neg_P","Neg_H"],
    "NegP":["Neg","QP_O"],
    "N_P_S":[], "N_H_S":[], "N_S":["N_P_S","N_H_S"],
    "Adj_P_S":[], "Adj_H_S":[], "Adj_S":["Adj_P_S","Adj_H_S"],
    "NP_S":["Adj_S","N_S"],
    "Q_P_S":[], "Q_H_S":[], "Q_S":["Q_P_S","Q_H_S"],
    "QP_S":["Q_S","NP_S","NegP"],
}

Q_VALUES = [determiner_signatures[q1][q2] for q1 in determiner_signatures for q2 in determiner_signatures[q1]]
values = {
    "N_P_O":["tree","rock"], "N_H_O":["tree","rock"], "N_O":[EQV,IND],
    "Adj_P_O":["happy","sad",EMPTY], "Adj_H_O":["happy","sad",EMPTY], "Adj_O":[EQV,IND,ENT,REV],
    "NP_O":[EQV,IND,ENT,REV],
    "Q_P_O":["some","every"], "Q_H_O":["some","every"], "Q_O":Q_VALUES,
    "V_P":["climbed","threw"], "V_H":["climbed","threw"], "V":[EQV,IND],
    "Adv_P":["energetically","joyfully",EMPTY], "Adv_H":["energetically","joyfully",EMPTY], "Adv":[EQV,IND,ENT,REV],
    "VP":[EQV,IND,ENT,REV], "QP_O":[EQV,IND,ENT,REV],
    "Neg_P":["not",EMPTY], "Neg_H":["not",EMPTY],
    "Neg":[negation_signature, emptystring_signature, compose_contradiction_signature, compose_neg_contradiction_signature],
    "NegP":RELATIONS,
    "N_P_S":["child","dog"], "N_H_S":["child","dog"], "N_S":[EQV,IND],
    "Adj_P_S":["little","cute",EMPTY], "Adj_H_S":["little","cute",EMPTY], "Adj_S":[EQV,IND,ENT,REV],
    "NP_S":[EQV,IND,ENT,REV],
    "Q_P_S":["some","every"], "Q_H_S":["some","every"], "Q_S":Q_VALUES,
    "QP_S":RELATIONS,
}

def adj_merge(adj_p, adj_h):
    if adj_p==adj_h: return EQV
    if adj_p==EMPTY: return REV
    if adj_h==EMPTY: return ENT
    return IND
adv_merge = adj_merge
def noun_phrase(adj, noun): return adj if noun==EQV else IND
verb_phrase = noun_phrase
def negation_merge(neg_p, neg_h):
    if neg_p==neg_h and neg_p==EMPTY: return Hashabledict(emptystring_signature)
    if neg_p==neg_h and neg_p!=EMPTY: return Hashabledict(negation_signature)
    if neg_p==EMPTY: return Hashabledict(compose_contradiction_signature)
    return Hashabledict(compose_neg_contradiction_signature)
negation_phrase = lambda neg, qp: neg[qp]
noun_merge = lambda a,b: EQV if a==b else IND
verb_merge = noun_merge
quantifier_merge = lambda q_p,q_h: determiner_signatures[q_p][q_h]
quantifier_phrase = lambda q,np,vp: q[np][vp]

functions = {
    "N_P_O":lambda:"tree", "N_H_O":lambda:"tree", "N_O":noun_merge,
    "Adj_P_O":lambda:"happy", "Adj_H_O":lambda:"happy", "Adj_O":adj_merge, "NP_O":noun_phrase,
    "Q_P_O":lambda:"some", "Q_H_O":lambda:"some", "Q_O":quantifier_merge,
    "V_P":lambda:"climbed", "V_H":lambda:"climbed", "V":verb_merge,
    "Adv_P":lambda:"energetically", "Adv_H":lambda:"energetically", "Adv":adv_merge,
    "VP":verb_phrase, "QP_O":quantifier_phrase,
    "Neg_P":lambda:"not", "Neg_H":lambda:"not", "Neg":negation_merge, "NegP":negation_phrase,
    "N_P_S":lambda:"dog", "N_H_S":lambda:"dog", "N_S":noun_merge,
    "Adj_P_S":lambda:"cute", "Adj_H_S":lambda:"cute", "Adj_S":adj_merge, "NP_S":noun_phrase,
    "Q_P_S":lambda:"some", "Q_H_S":lambda:"some", "Q_S":quantifier_merge, "QP_S":quantifier_phrase,
}
pos = {v:(i,0) for i,v in enumerate(parents)}  # layout only; not used numerically
variables = list(parents.keys())
mqnli_model = CausalModel(variables, values, parents, functions, pos=pos)
print("MQNLI CausalModel built.  #variables =", len(variables))
print("Root / output variable     :", mqnli_model.outputs)
print("Answer vocabulary (RELATIONS):", RELATIONS)

# Sanity: one factual example + one interchange intervention on NegP.
ex = mqnli_model.sample_input_tree_balanced()
setting = mqnli_model.run_forward(ex)
def _prem(s): return " ".join(str(s[k]) for k in ["Q_P_S","Adj_P_S","N_P_S","Neg_P","Adv_P","V_P","Q_P_O","Adj_P_O","N_P_O"])
def _hyp(s):  return " ".join(str(s[k]) for k in ["Q_H_S","Adj_H_S","N_H_S","Neg_H","Adv_H","V_H","Q_H_O","Adj_H_O","N_H_O"])
print("\nExample base:")
print("  premise   :", _prem(setting))
print("  hypothesis:", _hyp(setting))
print("  NegP =", setting["NegP"], "| QP_S (label) =", setting["QP_S"])
src = mqnli_model.sample_input_tree_balanced(output_var="NegP", output_var_value=CON)
inter = mqnli_model.run_interchange(ex, {"NegP": src})
print(f"\nAfter do(NegP := {src['NegP'] if 'NegP' in src else 'contradiction (from source)'}): QP_S ->", inter["QP_S"])


---
## 10. Fine-tuning GPT-2 on MQNLI

We now train a GPT-2 language model to solve MQNLI, exactly as the tutorial does:

- **Input format** (one fixed string per example):
  `Premise: <9 words>\nHypothesis: <9 words>\nRelation: ` with empty modifiers dropped.
- **Target**: the first sub-word token of the relation word for `QP_S`.
- **Fixed-length tokenization** (`padding="max_length", max_length=64`); the gold token is
  placed at the **last index**, so the model is trained to emit the answer at the fixed
  decision position. This is what later lets DAS intervene at one stable site (`MAX_LENGTH-2`).
- Standard causal-LM shift loss via HuggingFace `Trainer`.

**Factual gate.** DAS is only meaningful once the model actually solves the task. We require
the held-out accuracy to clear a gate (0.90 at project scale) before trusting any DAS number.

> `RUN_SMOKE_TEST = True` gives a ~1-minute debug pass (low accuracy, just checks the code
> runs). Set it to `False` for the real Colab-Pro run that clears the gate.


In [ ]:
# ── Section 10: factual MQNLI GPT-2 (faithful create_gpt2_lm training) ──────
import os
os.environ["WANDB_DISABLED"] = "true"
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from transformers import TrainingArguments, Trainer
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, classification_report
from pyvene.models.gpt2.modelings_intervenable_gpt2 import create_gpt2_lm
from tqdm.auto import tqdm
try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

RUN_SMOKE_TEST = False   # Colab-Pro full run.  Set True for a fast debug pass.
DAS_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if RUN_SMOKE_TEST:
    EXP = dict(factual_train_n=300, factual_test_n=100, factual_epochs=1, factual_lr=5e-5,
               factual_batch_size=8, das_train_n=100, das_test_n=100, das_epochs=1, das_lr=1e-3,
               das_batch_size=8, das_subspace_dim=32, das_layer=10, max_length=64, factual_gate=0.80)
else:
    EXP = dict(factual_train_n=5000, factual_test_n=1000, factual_epochs=5, factual_lr=5e-5,
               factual_batch_size=16, das_train_n=1000, das_test_n=1000, das_epochs=3, das_lr=1e-3,
               das_batch_size=16, das_subspace_dim=128, das_layer=10, max_length=64, factual_gate=0.90)
MAX_LENGTH = EXP["max_length"]
IGNORE_INDEX = -100
print("EXP:", EXP)

# ── text + tokenization helpers ──────────────────────────────────────────────
def clean_join(toks): return " ".join(str(t) for t in toks if str(t).strip() != "")
_PREM_KEYS = ["Q_P_S","Adj_P_S","N_P_S","Neg_P","Adv_P","V_P","Q_P_O","Adj_P_O","N_P_O"]
_HYP_KEYS  = ["Q_H_S","Adj_H_S","N_H_S","Neg_H","Adv_H","V_H","Q_H_O","Adj_H_O","N_H_O"]
def premise_to_string(s):    return clean_join([s[k] for k in _PREM_KEYS])
def hypothesis_to_string(s): return clean_join([s[k] for k in _HYP_KEYS])
def preprocess_input(s):     return f"Premise: {premise_to_string(s)}\nHypothesis: {hypothesis_to_string(s)}\nRelation: "
def preprocess_output(s, out="QP_S"): return s[out]

def set_pad(tok):
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    return tok
def tok_texts(tok, texts, max_length=MAX_LENGTH):
    return tok(texts, padding="max_length", max_length=max_length, truncation=True, return_tensors="pt")
def first_label_token_id(tok, label):
    return tok(str(label), add_special_tokens=False)["input_ids"][0]
def relation_token_map(tok):
    m = {r: first_label_token_id(tok, r) for r in RELATIONS}
    inv = {v:k for k,v in m.items()}
    if len(inv) != len(m): print("WARNING: relation labels share a first token:", m)
    return m, inv

def preprocess_factual(X, y, tok, max_length=MAX_LENGTH, out="QP_S"):
    texts = [preprocess_input(x) for x in X]
    label_ids = [first_label_token_id(tok, preprocess_output(yy, out)) for yy in y]
    enc = tok_texts(tok, texts, max_length)
    enc["labels"] = torch.full_like(enc["input_ids"], IGNORE_INDEX)
    enc["labels"][:, -1] = torch.tensor(label_ids, dtype=torch.long)
    return enc

def generate_factual_splits(train_n, test_n, seed=42):
    random.seed(seed); np.random.seed(seed)
    tr = mqnli_model.generate_factual_dataset(train_n, sampler=mqnli_model.sample_input_tree_balanced, return_tensors=False)
    te = mqnli_model.generate_factual_dataset(test_n,  sampler=mqnli_model.sample_input_tree_balanced, return_tensors=False)
    return ([e["input_ids"] for e in tr], [e["labels"] for e in tr],
            [e["input_ids"] for e in te], [e["labels"] for e in te])

def make_training_args(**kw):
    try: return TrainingArguments(**kw)
    except TypeError:
        if "evaluation_strategy" in kw: kw["eval_strategy"] = kw.pop("evaluation_strategy")
        if "save_strategy" in kw: kw.pop("save_strategy", None)
        return TrainingArguments(**kw)

def make_accuracy_metric(tok):
    def metric(eval_pred):
        labels = eval_pred.label_ids[:, -1]
        # predictions are pre-reduced to argmax token ids by preprocess_logits_for_metrics.
        preds  = eval_pred.predictions[:, -2]
        return {"accuracy": accuracy_score(labels, preds)}
    return metric

def _argmax_logits(logits, labels):
    # Keep memory bounded: reduce (B, T, V) logits to (B, T) token ids before gathering.
    return logits.argmax(dim=-1)

# ── train the factual model ──────────────────────────────────────────────────
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
mq_config, mq_tokenizer, mq_base_model = create_gpt2_lm()
mq_tokenizer = set_pad(mq_tokenizer)
REL2TOK, TOK2REL = relation_token_map(mq_tokenizer)
print("relation -> first-token id:", REL2TOK)

Xtr, ytr, Xte, yte = generate_factual_splits(EXP["factual_train_n"], EXP["factual_test_n"], seed=SEED)
print("train label dist:", dict(Counter(y["QP_S"] for y in ytr)))
print("test  label dist:", dict(Counter(y["QP_S"] for y in yte)))

train_ds = HFDataset.from_dict(preprocess_factual(Xtr, ytr, mq_tokenizer))
test_ds  = HFDataset.from_dict(preprocess_factual(Xte, yte, mq_tokenizer))

MQ_TRAIN_DIR = "mqnli_factual_gpt2"
args = make_training_args(
    output_dir=MQ_TRAIN_DIR, overwrite_output_dir=True,
    evaluation_strategy="epoch", save_strategy="no",
    learning_rate=EXP["factual_lr"], num_train_epochs=EXP["factual_epochs"],
    per_device_train_batch_size=EXP["factual_batch_size"],
    per_device_eval_batch_size=EXP["factual_batch_size"],
    report_to="none", use_cpu=(DAS_DEVICE=="cpu"), logging_steps=50, seed=SEED)
mq_trainer = Trainer(model=mq_base_model, args=args, train_dataset=train_ds,
                     eval_dataset=test_ds, compute_metrics=make_accuracy_metric(mq_tokenizer),
                     preprocess_logits_for_metrics=_argmax_logits)
mq_trainer.train()

# Held-out factual accuracy + per-relation report.
_pred = mq_trainer.predict(test_ds)
_lab  = _pred.label_ids[:, -1]
_prd  = _pred.predictions[:, -2]   # already argmaxed by preprocess_logits_for_metrics
mq_factual_acc = accuracy_score(_lab, _prd)
print(f"\nMQNLI factual held-out accuracy: {mq_factual_acc:.3f}  (gate {EXP['factual_gate']:.2f})")
print(classification_report([TOK2REL.get(int(t), f'tok{t}') for t in _lab],
                            [TOK2REL.get(int(t), f'tok{t}') for t in _prd], zero_division=0))
if mq_factual_acc < EXP["factual_gate"]:
    print("WARNING: factual accuracy below gate — DAS numbers below are debugging-only.")
MQNLI_READY = mq_factual_acc >= EXP["factual_gate"]


---
## 11. DAS on MQNLI internal variables

With a model that solves MQNLI, we ask **where** each high-level variable lives. For a target
variable `V` we:

1. Sample counterfactual data with `generate_counterfactual_dataset`: a base input, a **source**
   input chosen to realize a fresh random value of `V`, and the gold label
   `run_interchange(base, {V: source})`.
2. Wrap the model with a pyvene `IntervenableModel` carrying a **`RotatedSpaceIntervention`**
   at `block_output`, layer 10, position `MAX_LENGTH-2`, on a `subspace_dim`-dimensional
   rotated subspace.
3. Train **only the rotation** (model frozen) to make the patched run emit the counterfactual
   label. The held-out **counterfactual accuracy (IIA)** is the DAS metric.

We report three reference lines for every variable:

- **majority baseline** — always predicting the most frequent counterfactual label;
- **untrained rotation** — IIA of a random (untrained) rotated subspace at the same site;
- **DAS** — IIA after training the rotation.

Targets: **`QP_S`** (the root — a sanity check that the answer is decodable at the site) and
**`NegP`** (the tutorial's headline intermediate variable). We add two negative controls:
a **wrong-variable** control (train the rotation on `N_O` data, test on `NegP`) and a
**shuffled-label** control (train on `NegP` pairs with permuted counterfactual labels).


In [ ]:
# ── Section 11a: counterfactual data + DAS core (faithful to the tutorial) ──
from pyvene import IntervenableModel, RotatedSpaceIntervention, RepresentationConfig, IntervenableConfig

def sample_variable_intervention(var):
    vals = list(mqnli_model.values[var])
    def _s(*a, **k): return {var: random.choice(vals)}
    return _s
def fixed_intervention_id(*a, **k): return 0

def generate_counterfactual_data(var, n, seed=42):
    random.seed(seed); np.random.seed(seed)
    return mqnli_model.generate_counterfactual_dataset(
        n, fixed_intervention_id, 1,
        sampler=mqnli_model.sample_input_tree_balanced,
        intervention_sampler=sample_variable_intervention(var),
        return_tensors=False)

def majority_cf_acc(raw):
    c = Counter(d["labels"]["QP_S"] for d in raw)
    return max(c.values()) / max(1, sum(c.values()))

def preprocess_counterfactual(data, tok, max_length=MAX_LENGTH):
    out = []
    for d in data:
        base_text = preprocess_input(d["input_ids"])
        src_text  = preprocess_input(d["source_input_ids"][0])
        lid  = first_label_token_id(tok, d["labels"]["QP_S"])
        blid = first_label_token_id(tok, d["base_labels"]["QP_S"])
        item = {}
        item["input"]  = tok_texts(tok, base_text, max_length)
        item["source"] = [tok_texts(tok, src_text, max_length)]
        item["label"]      = torch.full_like(item["input"]["input_ids"], IGNORE_INDEX); item["label"][:, -1] = lid
        item["base_label"] = torch.full_like(item["input"]["input_ids"], IGNORE_INDEX); item["base_label"][:, -1] = blid
        out.append(item)
    return out

def shuffle_counterfactual_labels(data, seed=0):
    rng = random.Random(seed); sh = copy.deepcopy(data)
    labels = [it["label"].clone() for it in sh]; rng.shuffle(labels)
    for it, lb in zip(sh, labels): it["label"] = lb
    return sh

def make_intervenable_model(model, layer, subspace_dim, rep="block_output"):
    cfg = IntervenableConfig(
        model_type=type(model),
        representations=[RepresentationConfig(layer, rep, "pos", 1, subspace_partition=[[0, subspace_dim]])],
        intervention_types=RotatedSpaceIntervention)
    iv = IntervenableModel(cfg, model, use_fast=True)
    if DAS_DEVICE == "cuda": iv.set_device("cuda")
    iv.disable_model_gradients()
    return iv

def intervention_location(bs, position=None):
    if position is None: position = MAX_LENGTH - 2
    return {"sources->base": ([[[position]] * bs], [[[position]] * bs])}

def das_loss(logits, labels):
    sl = logits[..., :-1, :].contiguous(); sy = labels[..., 1:].contiguous()
    return CrossEntropyLoss()(sl.view(-1, sl.size(-1)), sy.view(-1))

def das_optimizer(iv, lr):
    params = []
    for _, it in iv.interventions.items():
        params += list(it.rotate_layer.parameters()) if hasattr(it, "rotate_layer") else list(it.parameters())
    if not params: raise ValueError("no trainable rotation parameters")
    return torch.optim.Adam(params, lr=lr)

def evaluate_intervenable(iv, dataset, batch_size, position=None):
    iv.model.eval(); preds=[]; labs=[]
    with torch.no_grad():
        for batch in DataLoader(dataset, batch_size=batch_size):
            bs = batch["input"]["input_ids"].shape[0]
            inputs  = {k: v.to(iv.get_device()) for k, v in batch["input"].items()}
            sources = [{k: v.to(iv.get_device()) for k, v in s.items()} for s in batch["source"]]
            _, cf = iv(inputs, sources, intervention_location(bs, position), subspaces=[[[0]] * bs])
            preds.append(cf.logits.argmax(-1)[..., -2].detach().cpu())
            labs.append(batch["label"][..., -1].detach().cpu())
    return accuracy_score(torch.cat(labs).numpy(), torch.cat(preds).numpy())

def train_intervenable(iv, dataset, epochs, lr, batch_size, position=None):
    opt = das_optimizer(iv, lr)
    for ep in range(int(epochs)):
        iv.model.train()
        it = tqdm(DataLoader(dataset, batch_size=batch_size, drop_last=True), desc=f"DAS epoch {ep}")
        for batch in it:
            bs = batch["input"]["input_ids"].shape[0]
            inputs  = {k: v.to(iv.get_device()) for k, v in batch["input"].items()}
            sources = [{k: v.to(iv.get_device()) for k, v in s.items()} for s in batch["source"]]
            _, cf = iv(inputs, sources, intervention_location(bs, position), subspaces=[[[0]] * bs])
            labels = batch["label"].to(iv.get_device())
            loss = das_loss(cf.logits, labels)
            opt.zero_grad(); loss.backward(); opt.step()
            if hasattr(iv, "set_zero_grad"): iv.set_zero_grad()
            it.set_postfix(loss=float(loss.detach().cpu()))
    return iv

def train_single_das(var, layer=None, subspace_dim=None, position=None,
                     train_n=None, test_n=None, seed=42, shuffle_labels=False,
                     train_raw=None, test_raw=None):
    layer        = EXP["das_layer"]        if layer is None else layer
    subspace_dim = EXP["das_subspace_dim"] if subspace_dim is None else subspace_dim
    train_n = train_n or EXP["das_train_n"]; test_n = test_n or EXP["das_test_n"]
    if train_raw is None: train_raw = generate_counterfactual_data(var, train_n, seed=seed)
    if test_raw  is None: test_raw  = generate_counterfactual_data(var, test_n,  seed=seed + 1)
    tr = preprocess_counterfactual(train_raw, mq_tokenizer)
    te = preprocess_counterfactual(test_raw,  mq_tokenizer)
    if shuffle_labels: tr = shuffle_counterfactual_labels(tr, seed=seed)
    iv = make_intervenable_model(copy.deepcopy(mq_base_model), layer, subspace_dim)
    untrained = evaluate_intervenable(iv, te, EXP["das_batch_size"], position)
    iv = train_intervenable(iv, tr, EXP["das_epochs"], EXP["das_lr"], EXP["das_batch_size"], position)
    das = evaluate_intervenable(iv, te, EXP["das_batch_size"], position)
    row = {"variable": var, "layer": layer, "subspace_dim": subspace_dim,
           "position": MAX_LENGTH-2 if position is None else position,
           "majority_baseline": majority_cf_acc(test_raw),
           "untrained_rotation": untrained, "das_accuracy": das,
           "shuffle_labels": shuffle_labels}
    print(pd.Series(row)); return row, iv


In [ ]:
# ── Section 11b: run DAS for QP_S and NegP + two negative controls ──────────
mqnli_das_rows = []

print("="*70, "\nDAS target: QP_S (root label — sanity check)\n", "="*70, sep="")
qps_row, qps_iv = train_single_das("QP_S", seed=SEED)
mqnli_das_rows.append(qps_row)

print("\n", "="*70, "\nDAS target: NegP (headline intermediate variable)\n", "="*70, sep="")
negp_row, negp_iv = train_single_das("NegP", seed=SEED + 10)
mqnli_das_rows.append(negp_row)

# Control 1 — wrong variable: train rotation on N_O data, evaluate on NegP data.
print("\n", "="*70, "\nControl: train on N_O, evaluate on NegP (wrong-variable)\n", "="*70, sep="")
_wrong_train = generate_counterfactual_data("N_O",  EXP["das_train_n"], seed=SEED + 20)
_negp_test   = generate_counterfactual_data("NegP", EXP["das_test_n"],  seed=SEED + 21)
wrong_row, _ = train_single_das("NegP", seed=SEED + 20,
                                train_raw=_wrong_train, test_raw=_negp_test)
wrong_row["variable"] = "NegP (train=N_O)"; wrong_row["control"] = "wrong_variable"
mqnli_das_rows.append(wrong_row)

# Control 2 — shuffled labels on NegP.
print("\n", "="*70, "\nControl: NegP with shuffled counterfactual labels\n", "="*70, sep="")
shuf_row, _ = train_single_das("NegP", seed=SEED + 30, shuffle_labels=True)
shuf_row["variable"] = "NegP (shuffled labels)"; shuf_row["control"] = "shuffled_labels"
mqnli_das_rows.append(shuf_row)

mqnli_df = pd.DataFrame(mqnli_das_rows)
mqnli_df.to_csv("outputs/tables/mqnli_das.csv", index=False)
display(mqnli_df[["variable","layer","subspace_dim","majority_baseline","untrained_rotation","das_accuracy"]].round(3))

fig, ax = plt.subplots(figsize=(8, 4))
_x = np.arange(len(mqnli_df)); _w = 0.35
ax.bar(_x - _w/2, mqnli_df["das_accuracy"],     _w, label="DAS (trained rotation)", color="tab:blue")
ax.bar(_x + _w/2, mqnli_df["untrained_rotation"], _w, label="untrained rotation", color="tab:gray")
ax.plot(_x, mqnli_df["majority_baseline"], "k_", ms=22, mew=2, label="majority baseline")
ax.set_xticks(_x); ax.set_xticklabels(mqnli_df["variable"], rotation=20, ha="right")
ax.set_ylabel("counterfactual accuracy (IIA)"); ax.set_ylim(0, 1.05)
ax.set_title(f"MQNLI DAS @ L{EXP['das_layer']} pos {MAX_LENGTH-2}, dim {EXP['das_subspace_dim']}")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig("outputs/figures/mqnli_das.png", dpi=150, bbox_inches="tight"); plt.show()

NEGP_IIA = float(negp_row["das_accuracy"])
print(f"\nNegP DAS IIA = {NEGP_IIA:.3f}  vs wrong-variable {wrong_row['das_accuracy']:.3f}, "
      f"shuffled {shuf_row['das_accuracy']:.3f}, majority {negp_row['majority_baseline']:.3f}")


---
## 12. Composition — two simultaneous interchange interventions

The project's extension beyond the tutorial: can a single layer host **two** disentangled
high-level variables that **compose**? We intervene on two variables at once, each from its
own source, and ask whether the model emits the **jointly** computed counterfactual label
`run_interchange(base, {V1: src_A, V2: src_B})`.

**Which two variables?** They must lie on **independent branches** of the graph that both feed
the root. `NegP` and `QP_O` would be a *degenerate* choice: `QP_O` is an **ancestor** of `NegP`
(`NegP = negation_phrase(Neg, QP_O)`), so forcing `NegP` overrides any `do(QP_O)` and the
second intervention has no effect on the output. Instead we compose **`NegP`** (object/negation
branch) with **`NP_S`** (subject branch) — both are direct parents of `QP_S` on disjoint
sub-trees, so the joint do-operation is non-trivial.

**Mechanism (pyvene multi-intervention).** We place two `RotatedSpaceIntervention`s at the same
site (layer 10, position `MAX_LENGTH-2`) on **non-overlapping subspace partitions**
(`[0, d]` for `NegP`, `[d, 2d]` for `NP_S`). Both rotations are trained **jointly** on joint
counterfactual data; the orthogonal partitions force the two variables into separate subspaces.
High joint IIA means the two interventions do not interfere — evidence of compositional,
localized structure.


In [ ]:
# ── Section 12: joint do(NegP=src_A, NP_S=src_B) — two-intervention DAS ─────
COMP_VARS = ("NegP", "NP_S")
COMP_LAYER = EXP["das_layer"]
COMP_DIM   = EXP["das_subspace_dim"] // 2          # each variable gets half the subspace
COMP_POS   = MAX_LENGTH - 2

def generate_joint_cf_data(v1, v2, n, seed=42):
    """Counterfactual data with a simultaneous intervention on v1 and v2 (separate sources)."""
    random.seed(seed); np.random.seed(seed)
    vals1 = list(mqnli_model.values[v1]); vals2 = list(mqnli_model.values[v2])
    def sampler(*a, **k): return {v1: random.choice(vals1), v2: random.choice(vals2)}
    return mqnli_model.generate_counterfactual_dataset(
        n, fixed_intervention_id, 1, sampler=mqnli_model.sample_input_tree_balanced,
        intervention_sampler=sampler, return_tensors=False)

def preprocess_joint(data, tok, max_length=MAX_LENGTH):
    """source_input_ids[0] realizes v1, [1] realizes v2 (order follows variable order in graph)."""
    out = []
    for d in data:
        item = {}
        item["input"]  = tok_texts(tok, preprocess_input(d["input_ids"]), max_length)
        item["source"] = [tok_texts(tok, preprocess_input(d["source_input_ids"][0]), max_length),
                           tok_texts(tok, preprocess_input(d["source_input_ids"][1]), max_length)]
        lid = first_label_token_id(tok, d["labels"]["QP_S"])
        item["label"] = torch.full_like(item["input"]["input_ids"], IGNORE_INDEX); item["label"][:, -1] = lid
        out.append(item)
    return out

def make_joint_intervenable(model, layer, dim, rep="block_output"):
    part = [[0, dim], [dim, 2 * dim]]
    cfg = IntervenableConfig(
        model_type=type(model),
        representations=[RepresentationConfig(layer, rep, "pos", 1, subspace_partition=part),
                         RepresentationConfig(layer, rep, "pos", 1, subspace_partition=part)],
        intervention_types=RotatedSpaceIntervention)
    iv = IntervenableModel(cfg, model, use_fast=True)
    if DAS_DEVICE == "cuda": iv.set_device("cuda")
    iv.disable_model_gradients()
    return iv

def joint_location(bs):
    return {"sources->base": ([[[COMP_POS]] * bs, [[COMP_POS]] * bs],
                              [[[COMP_POS]] * bs, [[COMP_POS]] * bs])}
def joint_subspaces(bs):
    # intervention 0 -> partition 0 ([0,dim]); intervention 1 -> partition 1 ([dim,2dim]).
    return [[[0]] * bs, [[1]] * bs]

def evaluate_joint(iv, dataset, batch_size):
    iv.model.eval(); preds=[]; labs=[]
    with torch.no_grad():
        for batch in DataLoader(dataset, batch_size=batch_size):
            bs = batch["input"]["input_ids"].shape[0]
            inputs  = {k: v.to(iv.get_device()) for k, v in batch["input"].items()}
            sources = [{k: v.to(iv.get_device()) for k, v in s.items()} for s in batch["source"]]
            _, cf = iv(inputs, sources, joint_location(bs), subspaces=joint_subspaces(bs))
            preds.append(cf.logits.argmax(-1)[..., -2].detach().cpu())
            labs.append(batch["label"][..., -1].detach().cpu())
    return accuracy_score(torch.cat(labs).numpy(), torch.cat(preds).numpy())

def train_joint(iv, dataset, epochs, lr, batch_size):
    opt = das_optimizer(iv, lr)
    for ep in range(int(epochs)):
        iv.model.train()
        it = tqdm(DataLoader(dataset, batch_size=batch_size, drop_last=True), desc=f"joint DAS ep {ep}")
        for batch in it:
            bs = batch["input"]["input_ids"].shape[0]
            inputs  = {k: v.to(iv.get_device()) for k, v in batch["input"].items()}
            sources = [{k: v.to(iv.get_device()) for k, v in s.items()} for s in batch["source"]]
            _, cf = iv(inputs, sources, joint_location(bs), subspaces=joint_subspaces(bs))
            loss = das_loss(cf.logits, batch["label"].to(iv.get_device()))
            opt.zero_grad(); loss.backward(); opt.step()
            if hasattr(iv, "set_zero_grad"): iv.set_zero_grad()
            it.set_postfix(loss=float(loss.detach().cpu()))
    return iv

print(f"Composition: do({COMP_VARS[0]}=src_A, {COMP_VARS[1]}=src_B) "
      f"@ L{COMP_LAYER}, pos {COMP_POS}, {COMP_DIM}-dim each")
_jtr = generate_joint_cf_data(*COMP_VARS, EXP["das_train_n"], seed=SEED + 40)
_jte = generate_joint_cf_data(*COMP_VARS, EXP["das_test_n"],  seed=SEED + 41)
joint_majority = majority_cf_acc(_jte)
print("joint counterfactual label dist:", dict(Counter(d["labels"]["QP_S"] for d in _jte)))
print(f"joint majority baseline: {joint_majority:.3f}")

jtr = preprocess_joint(_jtr, mq_tokenizer); jte = preprocess_joint(_jte, mq_tokenizer)
comp_iv = make_joint_intervenable(copy.deepcopy(mq_base_model), COMP_LAYER, COMP_DIM)
joint_untrained = evaluate_joint(comp_iv, jte, EXP["das_batch_size"])
comp_iv = train_joint(comp_iv, jtr, EXP["das_epochs"], EXP["das_lr"], EXP["das_batch_size"])
joint_das = evaluate_joint(comp_iv, jte, EXP["das_batch_size"])

comp_df = pd.DataFrame([
    {"condition": f"{COMP_VARS[0]} alone",  "IIA": float(negp_row["das_accuracy"])},
    {"condition": "joint untrained",        "IIA": joint_untrained},
    {"condition": "joint majority",         "IIA": joint_majority},
    {"condition": f"joint do({COMP_VARS[0]},{COMP_VARS[1]})", "IIA": joint_das},
]).round(3)
comp_df.to_csv("outputs/tables/composition.csv", index=False)
display(comp_df)

fig, ax = plt.subplots(figsize=(6.5, 4))
_c = ["tab:blue","tab:gray","black","tab:green"]
ax.bar(comp_df["condition"], comp_df["IIA"], color=_c)
ax.set_ylabel("counterfactual accuracy (IIA)"); ax.set_ylim(0, 1.05)
ax.set_title("Composition: jointly intervening on two independent-branch variables")
plt.xticks(rotation=20, ha="right"); fig.tight_layout()
fig.savefig("outputs/figures/composition.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"\nJoint do({COMP_VARS[0]},{COMP_VARS[1]}) IIA = {joint_das:.3f} "
      f"(untrained {joint_untrained:.3f}, majority {joint_majority:.3f})")


---
## 13. Calibration — random-init control + summary

DAS optimizes a rotation, so we must bound how much IIA comes from a *flexible alignment map*
rather than learned model structure (the Makelov / Sutter "alignment capacity" critique). We
re-run the **NegP** search at the same site on a **randomly-initialized** GPT-2 (`create_gpt2_lm`
with a fresh config, no factual training). If the random model's IIA is near the majority
baseline, the fine-tuned NegP result reflects learned structure; if it is close to the
fine-tuned IIA, the rotation is too expressive to interpret.

The final cell also collects every saved result into one summary table.


In [ ]:
# ── Section 13: random-init calibration for NegP + final summary ───────────
from transformers import GPT2Config, GPT2LMHeadModel

# Fresh, untrained GPT-2 with the same architecture (no pretraining, no fine-tuning).
torch.manual_seed(SEED + 999)
_rand_cfg = GPT2Config.from_pretrained("gpt2")
if hasattr(_rand_cfg, "_attn_implementation"): _rand_cfg._attn_implementation = "eager"
rand_model = GPT2LMHeadModel(_rand_cfg)
for p in rand_model.parameters(): p.requires_grad_(False)

# Re-run the NegP DAS search on the random model at the same site/dim.
_negp_tr = generate_counterfactual_data("NegP", EXP["das_train_n"], seed=SEED + 10)
_negp_te = generate_counterfactual_data("NegP", EXP["das_test_n"],  seed=SEED + 11)
_rtr = preprocess_counterfactual(_negp_tr, mq_tokenizer)
_rte = preprocess_counterfactual(_negp_te, mq_tokenizer)
rand_iv = make_intervenable_model(copy.deepcopy(rand_model), EXP["das_layer"], EXP["das_subspace_dim"])
rand_iv = train_intervenable(rand_iv, _rtr, EXP["das_epochs"], EXP["das_lr"], EXP["das_batch_size"])
rand_negp_iia = evaluate_intervenable(rand_iv, _rte, EXP["das_batch_size"])

calib_df = pd.DataFrame([
    {"model": "Fine-tuned GPT-2 (NegP DAS)", "IIA": float(negp_row["das_accuracy"])},
    {"model": "Random-init GPT-2 (NegP DAS)", "IIA": rand_negp_iia},
    {"model": "majority baseline",            "IIA": float(negp_row["majority_baseline"])},
]).round(3)
calib_df.to_csv("outputs/tables/calibration_mqnli.csv", index=False)
display(calib_df)

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.bar(calib_df["model"], calib_df["IIA"], color=["tab:blue","tab:red","tab:gray"])
ax.set_ylabel("NegP counterfactual accuracy (IIA)"); ax.set_ylim(0, 1.05)
ax.set_title("Calibration: NegP DAS on fine-tuned vs random-init GPT-2")
plt.xticks(rotation=15, ha="right"); fig.tight_layout()
fig.savefig("outputs/figures/calibration_mqnli.png", dpi=150, bbox_inches="tight"); plt.show()

_gap = max(0.0, float(negp_row["das_accuracy"]) - rand_negp_iia)
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"MQNLI factual accuracy        : {mq_factual_acc:.3f}  (gate {EXP['factual_gate']:.2f}, ready={MQNLI_READY})")
print(f"QP_S  DAS IIA  (root sanity)  : {qps_row['das_accuracy']:.3f}")
print(f"NegP  DAS IIA  (headline)     : {negp_row['das_accuracy']:.3f}  "
      f"[wrong-var {wrong_row['das_accuracy']:.3f}, shuffled {shuf_row['das_accuracy']:.3f}]")
print(f"Composition do(NegP,NP_S) IIA : {joint_das:.3f}  (majority {joint_majority:.3f})")
print(f"Random-init NegP IIA          : {rand_negp_iia:.3f}  -> learned-structure gap {_gap:.3f}")
print("\nFigures -> outputs/figures/, tables -> outputs/tables/")
